# Download and Screen the Down Syndrome Open-Access Corpus

Builds the Down syndrome open-access literature corpus end to end: seeds the candidate
set from an archived MEDLINE snapshot (or a live PMC query), applies the eligibility
criteria with a per-paper audit trail, enriches metadata through the production fetcher,
archives the primary documents (JATS XML and publisher PDF), and applies the INCLUDE
funding label. Every stage checkpoints to disk and is resume-safe.

## Design

- Deterministic input: `MEDLINE_SNAPSHOT` in the setup cell seeds Step 1 from the
  archived export `input/PMC_DownSynd_09_02_2026.txt`, so the corpus is reproducible
  from a frozen input rather than a live query that drifts as PMC grows. Set it to
  `None` to seed from the live PMC esearch instead.
- Corpus scope is the query plus the eligibility criteria alone. INCLUDE funding never
  admits a paper: the curated program spreadsheet is consulted only at the end (Part B)
  to label papers that already qualified. `SEED_REGISTRIES` can widen the candidate set
  to the INCLUDE and EMODS registries; any seed still faces the identical gate.
- The INCLUDE tag is binary and comes from one authoritative source,
  `INCLUDE_Publications_by_2025-11-14.xlsx`, carrying Condition, Grant Number, RFA/PA
  Number and Fiscal Year.
- Production code is reused, not reimplemented: the eligibility gate, the Crossref
  retraction check and the section parser come from `scripts/fetch_pmc_publications.py`.
- The raw primary documents are archived alongside the parsed JSON: the PMC JATS XML for
  every paper, and the publisher PDF where Europe PMC serves one.
- Nothing already on disk is fetched twice: earlier run directories are scanned first,
  and existing metadata JSONs, XMLs, PDFs and retraction verdicts are reused
  (hardlinked) instead of re-downloaded. Only genuinely new items touch the network.
- Every screened record receives a verdict and reason in `step2_screening_audit.csv`.
- All identifiers are handled as strings; the PMCID is read from the article's own
  identifier list, never from a cited reference's.

## Running the notebook

Run every cell top to bottom in the `dschat` kernel from `srv/nbs`. With the snapshot
input and reuse in place, a run over an already-downloaded corpus completes in minutes
and touches the network only for the PubMed publication-type fetch and the Europe PMC
PDF preflight. Expected funnel for the 2026-09-02 snapshot: 2,075 records, 1,891
eligible, 1,837 retained, 1,765 with at least one retrieval-usable section.

Each run writes to its own dated directory `output/ds_include_pipeline_<RUN_DATE>/`; set
`RUN_DATE` to an earlier date to resume that run instead of starting a new one.

Downstream: `2_embed_and_index_ds_papers.ipynb` embeds and indexes this build's papers
(selection `n_sections >= 4`) into the Milvus collection
`DS_INCLUDE_2026_09_02_DEEPINFRA_2048`, named for this notebook's input snapshot.

Run order: top to bottom. Parts B and C can be re-run alone to refresh INCLUDE tags.


In [1]:
# ============================ SETUP ============================
import csv, json, os, re, sqlite3, sys, time
from collections import Counter
from datetime import date, datetime, timezone
from pathlib import Path
from xml.etree import ElementTree as ET

import pandas as pd
import requests
from dotenv import load_dotenv

SRV = Path.cwd().parent if Path.cwd().name == "nbs" else Path.cwd()
assert (SRV / "scripts" / "fetch_pmc_publications.py").exists(), f"run from srv/nbs (SRV={SRV})"
load_dotenv(SRV / ".env")          # NCBI_EMAIL, CROSSREF_EMAIL, NCBI_API_KEY used by fpp
sys.path.insert(0, str(SRV / "scripts"))
import fetch_pmc_publications as fpp   # production gate, parser, retraction check

# ------------------------------ configuration ------------------------------
# Corpus scope. False (default) = the corpus IS the PMC query result after the Step 2 gate;
# the INCLUDE spreadsheet is consulted only at the very end, to label papers that already
# satisfied every criterion. True = also admit INCLUDE/EMODS registry PMIDs as candidates
# (they still face the identical gate). With the 2026-09-02 snapshot, False gives 1,891 eligible papers;
# True widens the candidate set with registry PMIDs that the MeSH query does not return
# (indexed under the comorbidity, or outside the strict open-access subset).
SEED_REGISTRIES = False

# Smoke test: process only the first N papers, so every cell can be proven end to end in
# minutes before committing to a multi-hour run. Set the PIPELINE_LIMIT environment variable
# (e.g. PIPELINE_LIMIT=25) or edit this line. 0 or unset means the full corpus.
LIMIT_PAPERS = int(os.getenv("PIPELINE_LIMIT", "0")) or None

# Retrieval only searches Results, Discussion, Methods and Conclusion; Abstract and
# Introduction are stored but excluded by the Milvus filter. Some PMC records (typically
# older ones) carry an abstract and nothing else, so they satisfy "has XML" yet contribute
# nothing searchable. False (default) keeps them, matching the literal rule "has full text";
# True drops them, making the corpus count equal the searchable count. This flag moves only the small set of
# abstract-only papers; the audit reports both counts.
REQUIRE_RETRIEVAL_SECTION = False

# Deterministic input snapshot. When set to a MEDLINE export file, Step 1 seeds the corpus
# from that archived file instead of a live PMC esearch, making the run reproducible from a
# frozen input (the live query drifts as PMC grows). The 2026-09-02 export reproduces the
# live corpus exactly (2,075 records -> 1,891 eligible -> 1,837 retained -> 1,765 searchable).
# Set to None to restore the original live-esearch behaviour.
MEDLINE_SNAPSHOT = SRV / "input" / "PMC_DownSynd_09_02_2026.txt"
# MEDLINE_SNAPSHOT = None        # live esearch (original v2 behaviour)

# Each run writes to its own date-stamped directory, so successive downloads stay
# side by side and are easy to compare. RUN_DATE is today by default; set it to an
# earlier YYYY-MM-DD to RESUME that run instead of starting a fresh one (every stage
# is checkpointed inside its own directory, so resuming skips work already done).
RUN_DATE = date.today().isoformat()
# RUN_DATE = "2026-09-02"        # example: continue an earlier run

RUNS_ROOT = SRV / "output"
existing = sorted(d.name for d in RUNS_ROOT.glob("ds_include_pipeline_*") if d.is_dir())
if existing:
    print("existing runs:", ", ".join(existing))

OUT_DIR  = RUNS_ROOT / f"ds_include_pipeline_{RUN_DATE}"
fresh    = not OUT_DIR.exists()
OUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR = OUT_DIR / "json_cache"; JSON_DIR.mkdir(exist_ok=True)
XML_DIR  = OUT_DIR / "xml"; XML_DIR.mkdir(exist_ok=True)     # raw PMC JATS XML per paper
PDF_DIR  = OUT_DIR / "pdf"; PDF_DIR.mkdir(exist_ok=True)     # publisher PDF per paper

# Never download a document twice: earlier run directories are scanned before any
# network fetch in Steps 4a/4c, and an XML/PDF already on disk is hardlinked (or
# copied) into this run instead of downloaded again. Newest runs are tried first.
REUSE_DOC_DIRS = sorted((d for d in RUNS_ROOT.glob("ds_include_pipeline_*")
                         if d.is_dir() and d != OUT_DIR), reverse=True)
if REUSE_DOC_DIRS:
    print("document reuse dirs:", ", ".join(d.name for d in REUSE_DOC_DIRS))

# Also reuse prior FETCH RESULTS: the enriched metadata JSONs (Step 3) and the Crossref
# retraction verdicts (Step 2b) from the newest earlier run. With this on, a full manual
# rerun touches the network only for the PubMed publication-type fetch and the Europe PMC
# PDF preflight, and completes in minutes. Set False to re-fetch metadata and re-verify
# retractions live (hours).
REUSE_PRIOR_RESULTS = True
print(f"run directory: {OUT_DIR.name}  ({'new' if fresh else 'resuming'})")
if LIMIT_PAPERS:
    print(f"*** SMOKE TEST MODE: only {LIMIT_PAPERS} papers will be processed ***")
# To reuse the 2,178 already-fetched production JSONs instead of refetching, point JSON_DIR at
# SRV / "output" / "DS_PMC_05_2026". Caution: ingest_pmids.py treats presence in that cache as
# "already ingested", so only do this deliberately.
DB_PATH  = OUT_DIR / "include_tracker.db"

EMAIL   = fpp.NCBI_EMAIL
EUTILS  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
HEADERS = {"User-Agent": f"DS-INCLUDE-pipeline-v2 (mailto:{EMAIL})"}
SLEEP   = 0.34 if fpp.NCBI_API_KEY else 0.55


PMC_QUERY = ('"Down syndrome"[MeSH Terms] AND (open_access[Filter]) '
             'AND (published_article[Filter]) AND (pmc_public[Filter])')
# esearch translates this to
#   "Down syndrome"[MeSH Terms] AND "open access"[Filter] AND "published article"[Filter]
#   AND "pmc public"[Filter]        (2,075 records, no warnings)

def _request(method, url, *, params=None, data=None, timeout=90, tries=6, base_wait=3):
    """HTTP with retries; raises on final failure (never returns None silently)."""
    last = None
    for attempt in range(tries):
        try:
            r = requests.request(method, url, params=params, data=data,
                                 headers=HEADERS, timeout=timeout)
            if r.status_code in (429, 500, 502, 503):
                last = f"HTTP {r.status_code}"
                time.sleep(min(120, base_wait * 2 ** attempt)); continue
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            last = str(e)[:200]
            time.sleep(min(120, base_wait * 2 ** attempt))
    raise RuntimeError(f"{method} {url} failed after {tries} tries: {last}")

def get_json(url, params=None):  return _request("GET", url, params=params).json()

def save_ckpt(name, obj):  (OUT_DIR / name).write_text(json.dumps(obj))
def load_ckpt(name, default):
    p = OUT_DIR / name
    return json.loads(p.read_text()) if p.exists() else default

print("Setup OK ->", OUT_DIR)


existing runs: ds_include_pipeline_2026-09-02
document reuse dirs: ds_include_pipeline_2026-09-02
run directory: ds_include_pipeline_2026-09-05  (new)
Setup OK -> /path/to/project/T21_research_assistant_v7/srv/output/ds_include_pipeline_2026-09-05


---
# Part A. Down syndrome open-access corpus

## Step 1. Seed the candidate set

The corpus is the PMC search, exactly as run on the website:

    "Down syndrome"[MeSH Terms] AND (open_access[Filter]) AND (published_article[Filter])
    AND (pmc_public[Filter])

By default Step 1 parses the archived MEDLINE export of this query
(`MEDLINE_SNAPSHOT`), which freezes the candidate set for reproducibility; with the
snapshot unset, the same query runs live through `esearch` (db=pmc) followed by
`esummary` for each hit's PMID and DOI.

Embargoed articles are excluded twice over: the query filters them, and an embargoed
article serves no OAI body, so it yields no sections and could never be embedded even if
one slipped through.

INCLUDE funding is not a way into the corpus. Papers qualify on the criteria alone, and
the INCLUDE spreadsheet is consulted at the end (Part B) to label the survivors.

`SEED_REGISTRIES` in the setup cell can widen the candidate set to the INCLUDE and EMODS
registries if that is ever wanted; any seed still faces the identical Step 2 gate.


In [2]:
# ---- Step 1a: corpus ID acquisition ----
# MEDLINE_SNAPSHOT set   -> parse the archived export (deterministic, offline)
# MEDLINE_SNAPSHOT None  -> live PMC esearch (paged; hard guard at the 10k retstart cap)
medline_rows = []
if MEDLINE_SNAPSHOT:
    assert Path(MEDLINE_SNAPSHOT).exists(), f"snapshot not found: {MEDLINE_SNAPSHOT}"
    _rec, _tag = None, None
    for _line in open(MEDLINE_SNAPSHOT, encoding="utf-8", errors="replace"):
        if _line.startswith("PMID- "):
            if _rec: medline_rows.append(_rec)
            _rec = {"pmc_uid": "", "pmcid": "", "pmid": _line[6:].strip(), "doi": ""}
            _tag = "PMID"; continue
        if _rec is None: continue
        _m = re.match(r"^([A-Z]{1,4})\s*- ", _line)
        if not _m: continue
        _tag, _val = _m.group(1), _line[6:].strip()
        if _tag == "PMC" and not _rec["pmcid"]:
            _rec["pmcid"] = _val if _val.upper().startswith("PMC") else f"PMC{_val}"
            _rec["pmc_uid"] = _rec["pmcid"][3:]
        elif _tag in ("AID", "LID") and _val.endswith("[doi]") and not _rec["doi"]:
            _rec["doi"] = _val[:-5].strip()
    if _rec: medline_rows.append(_rec)
    uids = [r["pmc_uid"] for r in medline_rows if r["pmc_uid"]]
    _n_nopmc = sum(1 for r in medline_rows if not r["pmcid"])
    print(f"MEDLINE snapshot {Path(MEDLINE_SNAPSHOT).name}: {len(medline_rows)} records"
          + (f" ({_n_nopmc} without a PMCID; the Step 2 gate decides those)" if _n_nopmc else ""))
else:
    uids, retstart = [], 0
    while True:
        params = {"db": "pmc", "term": PMC_QUERY, "retmode": "json",
                  "retstart": retstart, "retmax": 5000, "email": EMAIL}
        if fpp.NCBI_API_KEY: params["api_key"] = fpp.NCBI_API_KEY
        res = get_json(f"{EUTILS}/esearch.fcgi", params)["esearchresult"]
        if res.get("warninglist"): print("esearch WARNINGS:", res["warninglist"])
        ids = res.get("idlist", [])
        uids.extend(ids)
        total = int(res.get("count", 0))
        if total > 9500:
            raise RuntimeError(f"{total} results exceeds the esearch retstart cap; "
                               "slice the query by publication date ranges")
        retstart += len(ids)
        if retstart >= total or not ids: break
        time.sleep(SLEEP)
    print(f"PMC records matching query: {total}")

MEDLINE snapshot PMC_DownSynd_09_02_2026.txt: 2075 records


In [3]:
# ---- Step 1b: pmcid / pmid / doi id map (all strings) ----
# Snapshot mode: the map comes straight from the MEDLINE parse (DOI from AID/LID [doi]
# tags where present; the Step 2 efetch fills any gaps from PubMed). Live mode: esummary.
if MEDLINE_SNAPSHOT:
    rows = medline_rows
    save_ckpt("step1_idmap.json", rows)
    print(f"id map from MEDLINE snapshot: {len(rows)} records "
          f"({sum(1 for r in rows if r['doi'])} with a DOI)")
else:
    rows = load_ckpt("step1_idmap.json", [])
    done_uids = {r["pmc_uid"] for r in rows}
    todo = [u for u in uids if u not in done_uids]
    for i in range(0, len(todo), 200):
        chunk = todo[i:i+200]
        data = {"db": "pmc", "id": ",".join(chunk), "retmode": "json", "email": EMAIL}
        if fpp.NCBI_API_KEY: data["api_key"] = fpp.NCBI_API_KEY
        res = _request("POST", f"{EUTILS}/esummary.fcgi", data=data).json()["result"]
        for uid in res.get("uids", []):
            ids = {a["idtype"]: str(a["value"]) for a in res[uid].get("articleids", [])}
            rows.append({"pmc_uid": uid, "pmcid": f"PMC{uid}",
                         "pmid": ids.get("pmid", ""), "doi": ids.get("doi", "")})
        save_ckpt("step1_idmap.json", rows)
        print(f"  esummary: {min(i+200, len(todo))}/{len(todo)}", end="\r")
        time.sleep(SLEEP)
    print()
query_df = (pd.DataFrame(rows).astype(str)
            .replace({"pmid": {"": None}}).dropna(subset=["pmid"])
            .drop_duplicates("pmid").reset_index(drop=True))
print(f"PMC query seed: {len(query_df)} records with PMIDs")


id map from MEDLINE snapshot: 2075 records (2057 with a DOI)
PMC query seed: 2075 records with PMIDs


In [4]:
# ---- Step 1c: union with the registry seeds ----
def _pmid_column(df):
    """PMID column by content, not by name (EMODS calls it PublicationID)."""
    for c in df.columns:
        s = df[c].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
        if len(s) > 10 and s.str.fullmatch(r"\d{7,8}").mean() > 0.5:
            return c
    return None

seeds = {"pmc_query": set(query_df["pmid"])}
registry_specs = ([("include_excel", "INCLUDE_Publications_*.xlsx"),
                   ("emods_excel",   "EMODS_publications_extract*.xlsx")]
                  if SEED_REGISTRIES else [])
for label, pattern in registry_specs:
    path = next(SRV.joinpath("input").glob(pattern), None)
    if path is None:
        print(f"  {label}: file not found, skipped"); continue
    df = pd.read_excel(path)
    col = _pmid_column(df)
    if col is None:
        print(f"  {label}: no PMID-like column in {list(df.columns)[:6]}"); continue
    s = (df[col].dropna().astype(str).str.replace(r"\.0$", "", regex=True)
         .str.extract(r"(\d{7,8})")[0].dropna())
    seeds[label] = set(s)
    print(f"  {label}: {len(seeds[label])} PMIDs (column {col!r} in {path.name})")

all_pmids = sorted(set().union(*seeds.values()))
if not SEED_REGISTRIES:
    print("  SEED_REGISTRIES is False: corpus scope is the PMC query alone;")
    print("  the INCLUDE spreadsheet is applied at the end (Part B) as a label only.")
for k, v in seeds.items():
    print(f"    {k:14s}: {len(v):5d} | unique to it: {len(v - set().union(*(x for j, x in seeds.items() if j != k)))}")
if LIMIT_PAPERS:
    all_pmids = all_pmids[:LIMIT_PAPERS]
    print(f"  LIMIT_PAPERS={LIMIT_PAPERS}: SMOKE TEST, not a real corpus build")
print(f"Candidate corpus (union of all seeds): {len(all_pmids)} PMIDs")
pd.DataFrame({"pmid": all_pmids,
              "seed_sources": ["; ".join(k for k, v in seeds.items() if p in v)
                               for p in all_pmids]}
             ).to_csv(OUT_DIR / "step1_corpus_ids.csv", index=False)


  SEED_REGISTRIES is False: corpus scope is the PMC query alone;
  the INCLUDE spreadsheet is applied at the end (Part B) as a label only.
    pmc_query     :  2075 | unique to it: 2075
Candidate corpus (union of all seeds): 2075 PMIDs


## Step 2. Eligibility gate (metadata criteria)

Batched implementation of the production gate (`fpp.check_pmid_eligibility`), in the same
order it uses:

1. Publication-type filter via MEDLINE PublicationTypeList against
   `fpp.EXCLUDED_PUBLICATION_TYPES` (case reports, retracted publications, comments,
   editorials, letters, personal narratives, biographies, historical articles, proceedings).
   This also enforces "journal article" status and catches MEDLINE-tagged retractions.
2. PMC membership: a PMCID must exist.
3. Crossref retraction cross-check via `fpp._check_retraction_crossref`, which reads both
   update-to and the relation.retraction / is-retracted-by keys (the channel Springer/BMC and
   Wiley actually use). Independent of signal 1; a paper is excluded if either fires.

Embargo needs no explicit flag: an embargoed article serves no OAI body, so it produces no
sections and is excluded by the full-text criterion in Step 3b.

A paper is retained when it has a PMID, is in PMC, is not retracted, is not an excluded
publication type, and PMC serves usable full text for it. Nothing else is required. Missing
MeSH terms and missing author keywords never exclude a paper: no stage filters on either
field, and papers lacking them are retained whenever the criteria above are met.


In [5]:
# ---- Step 2a: batched PublicationTypeList + PMCID confirmation ----
pt_rows = load_ckpt("step2_pubtypes.json", [])
done = {r["pmid"] for r in pt_rows}
todo = [p for p in all_pmids if p not in done]
for i in range(0, len(todo), 200):
    chunk = todo[i:i+200]
    data = {"db": "pubmed", "id": ",".join(chunk), "retmode": "xml", "email": EMAIL}
    if fpp.NCBI_API_KEY: data["api_key"] = fpp.NCBI_API_KEY
    root = ET.fromstring(_request("POST", f"{EUTILS}/efetch.fcgi", data=data).content)
    for art in root.findall(".//PubmedArticle"):
        pmid = (art.findtext(".//MedlineCitation/PMID") or "").strip()
        pts = [pt.text.strip() for pt in art.findall(".//PublicationType") if pt.text]
        pmcid = doi = ""
        # Scope to the article's own id list: a bare .//ArticleId also matches the
        # ArticleIds of every cited reference, and the loop would end up assigning a
        # reference's PMCID to this article (verified on PMID 11178230, 54 references).
        for aid in art.findall("./PubmedData/ArticleIdList/ArticleId"):
            if aid.get("IdType") == "pmc" and aid.text:
                pmcid = aid.text.strip()
                if not pmcid.upper().startswith("PMC"): pmcid = f"PMC{pmcid}"
            elif aid.get("IdType") == "doi" and aid.text:
                doi = aid.text.strip()
        pt_rows.append({"pmid": pmid, "pub_types": "; ".join(pts),
                        "pmcid_pubmed": pmcid, "doi_pubmed": doi,
                        "excluded_types": "; ".join(sorted(
                            fpp.EXCLUDED_PUBLICATION_TYPES & set(pts)))})
    save_ckpt("step2_pubtypes.json", pt_rows)
    print(f"  efetch: {min(i+200, len(todo))}/{len(todo)}", end="\r")
    time.sleep(SLEEP)
print()
pt_df = pd.DataFrame(pt_rows).astype(str).drop_duplicates("pmid")
gate = (pd.DataFrame({"pmid": all_pmids})
        .merge(pt_df, on="pmid", how="left")
        .merge(query_df[["pmid", "pmcid", "doi"]], on="pmid", how="left"))
gate = gate.fillna("")
# PMCID/DOI from the PMC query where available, else from PubMed (registry-only seeds)
gate["pmcid"] = gate["pmcid"].where(gate["pmcid"].astype(bool), gate["pmcid_pubmed"])
gate["doi"]   = gate["doi"].where(gate["doi"].astype(bool), gate["doi_pubmed"])
gate["seed_sources"] = ["; ".join(k for k, v in seeds.items() if p in v) for p in gate["pmid"]]
n_missing = int((gate["pub_types"] == "").sum())
if n_missing:
    print(f"NOTE: {n_missing} PMIDs returned no PubMed record (excluded as unverifiable)")
print("Publication-type exclusions found:")
print(gate.loc[gate["excluded_types"] != "", "excluded_types"]
          .str.split("; ").explode().value_counts().to_string() or "  none")


  efetch: 2075/2075
Publication-type exclusions found:
excluded_types
Case Reports                    130
Letter                           34
Comment                          16
Editorial                         9
Retracted Publication             3
Historical Article                3
Conference Proceedings            3
Personal Narrative                2
Introductory Journal Article      1
Biography                         1


In [6]:
# ---- Step 2b: Crossref retraction cross-check (production function, checkpointed) ----
candidates = gate[(gate["excluded_types"] == "") & (gate["pmcid"] != "")].copy()
cr = load_ckpt("step2_crossref.json", {})
if REUSE_PRIOR_RESULTS and REUSE_DOC_DIRS:
    _prior = REUSE_DOC_DIRS[0] / "step2_crossref.json"
    if _prior.exists():
        _new = {p: v for p, v in json.loads(_prior.read_text()).items() if p not in cr}
        if _new:
            cr.update(_new)
            save_ckpt("step2_crossref.json", cr)
            print(f"Crossref verdicts reused from {REUSE_DOC_DIRS[0].name}: {len(_new)}")
todo = [(p, d) for p, d in zip(candidates["pmid"], candidates["doi"]) if p not in cr]
for n, (pmid, doi) in enumerate(todo, 1):
    cr[pmid] = fpp._check_retraction_crossref(doi, fpp.CROSSREF_EMAIL)
    if n % 25 == 0:
        save_ckpt("step2_crossref.json", cr)
        print(f"  Crossref: {n}/{len(todo)}", end="\r")
    time.sleep(0.15)
save_ckpt("step2_crossref.json", cr)
print()
candidates["retracted_crossref"] = candidates["pmid"].map(
    lambda p: bool(cr.get(p, {}).get("is_retracted")))
eligible = candidates[~candidates["retracted_crossref"]].reset_index(drop=True)
print(f"Eligible after the full gate: {len(eligible)} / {len(all_pmids)}")
print(f"  excluded by publication type : {(gate['excluded_types'] != '').sum()}")
print(f"  excluded for missing PMCID   : {((gate['excluded_types'] == '') & (gate['pmcid'] == '')).sum()}")
print(f"  excluded as retracted (CR)   : {int(candidates['retracted_crossref'].sum())}")
assert eligible["pmid"].is_unique, "duplicate PMIDs in the eligible set"
assert eligible.loc[eligible["pmcid"] != "", "pmcid"].is_unique, "duplicate PMCIDs"
print(f"uniqueness check: {len(eligible)} rows, "
      f"{eligible['pmid'].nunique()} distinct PMIDs, "
      f"{eligible['pmcid'].nunique()} distinct PMCIDs")
# Per-paper audit trail: every screened record, its verdict, and the reason.
audit = []
for _, r in gate.iterrows():
    pmid = r["pmid"]
    crr = cr.get(pmid, {})
    if r["pub_types"] == "":
        verdict, reason = "excluded", "no PubMed record"
    elif r["excluded_types"]:
        verdict, reason = "excluded", f"publication type: {r['excluded_types']}"
    elif not r["pmcid"]:
        verdict, reason = "excluded", "no PMCID (not in PMC)"
    elif crr.get("is_retracted"):
        verdict, reason = "excluded", f"retracted (CrossRef): {crr.get('retraction_doi')}"
    else:
        verdict, reason = "retained", ""
    audit.append({"pmid": pmid, "pmcid": r["pmcid"], "doi": r["doi"],
                  "verdict": verdict, "reason": reason,
                  "pub_types": r["pub_types"],
                  "retraction_screened": pmid in cr,
                  "retraction_note": crr.get("error") or ""})
audit_df = pd.DataFrame(audit)
audit_df.to_csv(OUT_DIR / "step2_screening_audit.csv", index=False)

print("\nscreening flow (PRISMA style):")
print(f"  records identified by the PMC query      : {len(query_df)}")
if SEED_REGISTRIES:
    print(f"  records added from registry seeds        : {len(all_pmids) - len(query_df)}")
print(f"  records screened                         : {len(audit_df)}")
for reason, k in audit_df.loc[audit_df['verdict'] == 'excluded', 'reason'].value_counts().items():
    print(f"    excluded: {reason[:58]:58s} {k}")
print(f"  records retained                         : {int((audit_df['verdict'] == 'retained').sum())}")
n_unscreened = int((~audit_df['retraction_screened']).sum())
if n_unscreened:
    print(f"  (retraction screening not reached for {n_unscreened}, excluded earlier in the gate)")
print(f"  audit trail -> step2_screening_audit.csv")

print("\neligible papers by seed source:")
print(eligible["seed_sources"].value_counts().to_string())
eligible.to_csv(OUT_DIR / "step2_eligible_ids.csv", index=False)


Crossref verdicts reused from ds_include_pipeline_2026-09-02: 1891

Eligible after the full gate: 1891 / 2075
  excluded by publication type : 184
  excluded for missing PMCID   : 0
  excluded as retracted (CR)   : 0
uniqueness check: 1891 rows, 1891 distinct PMIDs, 1891 distinct PMCIDs

screening flow (PRISMA style):
  records identified by the PMC query      : 2075
  records screened                         : 2075
    excluded: publication type: Case Reports                             125
    excluded: publication type: Letter                                   21
    excluded: publication type: Comment; Letter                          8
    excluded: publication type: Editorial                                6
    excluded: publication type: Comment                                  6
    excluded: publication type: Case Reports; Letter                     5
    excluded: publication type: Retracted Publication                    3
    excluded: publication type: Conference Proceedin

## Step 2c. PMC Open Access Subset annotation

Free-to-read in PMC and the redistributable Open Access Subset are not the same thing.
This step annotates `in_pmc_oa_subset` for every paper so the manuscript can state
whichever number it claims.

Leave `REQUIRE_OA_SUBSET = False` unless strict OA Subset membership is claimed:
turning it on drops papers that are open access but outside the strict subset.


In [7]:
REQUIRE_OA_SUBSET = False

r = _request("POST", f"{EUTILS}/epost.fcgi",
             data={"db": "pubmed", "id": ",".join(eligible["pmid"])})
root = ET.fromstring(r.content)
webenv, qk = root.findtext(".//WebEnv"), root.findtext(".//QueryKey")
time.sleep(SLEEP)
res = get_json(f"{EUTILS}/esearch.fcgi",
               {"db": "pubmed", "term": f'#{qk} AND "pubmed pmc open access"[filter]',
                "WebEnv": webenv, "usehistory": "y", "retmax": 10000, "retmode": "json"})
oa_set = set(res["esearchresult"].get("idlist", []))
eligible["in_pmc_oa_subset"] = eligible["pmid"].isin(oa_set)
n_oa = int(eligible["in_pmc_oa_subset"].sum())
print(f"In the strict PMC OA Subset: {n_oa} / {len(eligible)}")
if REQUIRE_OA_SUBSET:
    eligible = eligible[eligible["in_pmc_oa_subset"]].reset_index(drop=True)
    print(f"REQUIRE_OA_SUBSET on -> {len(eligible)} papers retained")


In the strict PMC OA Subset: 1891 / 1891


## Step 3. Metadata enrichment (production fetcher)

`fpp.fetch_and_save_metadata` runs the full OpenAlex -> NCBI efetch -> Crossref -> PMC OAI
chain and writes one production-schema JSON per PMID, using the fixed section parser
(recursive subsection capture with parent-bucket propagation, combined-heading handling,
positional fallback for thematic headings). The output is directly consumable by
`embed_and_insert.py` and `ingest_pmids.py`. Resume-safe: existing JSONs are skipped.


In [8]:
errors = []

# Reuse pass: metadata JSONs already fetched by an earlier run are hardlinked in
# (copy fallback) so only genuinely new papers hit the APIs.
if REUSE_PRIOR_RESULTS:
    _reused = 0
    for _p in eligible["pmid"]:
        _dest = JSON_DIR / f"publication_metadata_PMID_{_p}.json"
        if _dest.exists():
            continue
        for _rd in REUSE_DOC_DIRS:
            _srcf = _rd / "json_cache" / _dest.name
            if _srcf.exists() and _srcf.stat().st_size > 0:
                try:
                    os.link(_srcf, _dest)
                except OSError:
                    _dest.write_bytes(_srcf.read_bytes())
                _reused += 1
                break
    if _reused:
        print(f"metadata JSONs reused from disk (no refetch): {_reused}")

todo = [ (p, c) for p, c in zip(eligible["pmid"], eligible["pmcid"])
         if not (JSON_DIR / f"publication_metadata_PMID_{p}.json").exists() ]
print(f"To fetch: {len(todo)} (already cached: {len(eligible) - len(todo)})")
for n, (pmid, pmcid) in enumerate(todo, 1):
    try:
        fpp.fetch_and_save_metadata(pmid, out_dir=JSON_DIR, fetch_fulltext=True,
                                    overwrite=False, sleep_between=0.5)
    except Exception as exc:
        errors.append({"pmid": pmid, "error": str(exc)[:200]})
    if n % 25 == 0: print(f"  enrich: {n}/{len(todo)} (errors: {len(errors)})", end="\r")
print()
if errors: save_ckpt("step3_errors.json", errors)
print(f"Enrichment done. errors: {len(errors)}")


metadata JSONs reused from disk (no refetch): 1891


To fetch: 0 (already cached: 1891)

Enrichment done. errors: 0


## Step 3b. Full-text availability (final criterion)

A paper is considered only if PMC actually serves usable full text for it. This is the
last criterion, and it can only be evaluated here, after the OAI fetch: a PMCID existing
in PubMed does not guarantee the OAI record carries a body (embargoed, metadata-only, or
deposit-pending records have none).

Papers with no extractable target section are excluded and recorded in the audit rather
than being dropped silently later at the embedding step. The cell also reports how many
retained papers carry at least one of the four sections retrieval uses (Results,
Discussion, Methods, Conclusion); `REQUIRE_RETRIEVAL_SECTION` controls whether
abstract-only papers are kept in the corpus or dropped.

Absent MeSH terms and absent author keywords never exclude a paper: no stage filters on
either field.


In [9]:
TARGET = ["Abstract", "Introduction", "Methods", "Results", "Discussion", "Conclusion"]
RETRIEVAL_SECTIONS = ["Results", "Discussion", "Methods", "Conclusion"]   # what search uses
def present(v): return v is not None and str(v).strip() not in ("", "N/A", "NA", "n/a", "na")

fulltext_rows, retained_pmids = [], []
for pmid in eligible["pmid"]:
    f = JSON_DIR / f"publication_metadata_PMID_{pmid}.json"
    if not f.exists():
        fulltext_rows.append({"pmid": pmid, "n_sections": 0, "verdict": "excluded",
                              "reason": "metadata fetch failed"})
        continue
    secs = (json.loads(f.read_text()).get("sections") or {})
    n_sec = sum(1 for s_ in TARGET if present(secs.get(s_)))
    n_ret = sum(1 for s_ in RETRIEVAL_SECTIONS if present(secs.get(s_)))
    if n_sec == 0:
        fulltext_rows.append({"pmid": pmid, "n_sections": 0, "n_retrieval": 0,
                              "verdict": "excluded", "reason": "no full text in PMC XML"})
    elif REQUIRE_RETRIEVAL_SECTION and n_ret == 0:
        fulltext_rows.append({"pmid": pmid, "n_sections": n_sec, "n_retrieval": 0,
                              "verdict": "excluded",
                              "reason": "abstract only, no retrieval-usable section"})
    else:
        retained_pmids.append(pmid)
        fulltext_rows.append({"pmid": pmid, "n_sections": n_sec, "n_retrieval": n_ret,
                              "verdict": "retained", "reason": ""})
ft = pd.DataFrame(fulltext_rows)
n_excl = int((ft["verdict"] == "excluded").sum())
print(f"full-text check: {len(retained_pmids)} retained, {n_excl} excluded")
if n_excl:
    print(ft.loc[ft["verdict"] == "excluded", "reason"].value_counts().to_string())
print("\nretained papers by section count:")
print(ft.loc[ft["verdict"] == "retained", "n_sections"].value_counts().sort_index().to_string())
n_abs_only = int(((ft["verdict"] == "retained") & (ft["n_retrieval"] == 0)).sum())
n_search = int(((ft["verdict"] == "retained") & (ft["n_retrieval"] > 0)).sum())
print(f"\nsearchable (>=1 of Results/Discussion/Methods/Conclusion): {n_search}")
if n_abs_only:
    print(f"abstract-only, retained but never searched: {n_abs_only} "
          f"(set REQUIRE_RETRIEVAL_SECTION=True to drop them)")

# fold this final criterion into the screening audit
audit_df = (audit_df.merge(ft[["pmid", "verdict", "reason", "n_sections"]],
                           on="pmid", how="left", suffixes=("", "_ft")))
mask = audit_df["verdict_ft"] == "excluded"
audit_df.loc[mask, "verdict"] = "excluded"
audit_df.loc[mask, "reason"] = audit_df.loc[mask, "reason_ft"]
audit_df = audit_df.drop(columns=["verdict_ft", "reason_ft"])
audit_df.to_csv(OUT_DIR / "step2_screening_audit.csv", index=False)
print(f"\nFINAL corpus after all criteria: {len(retained_pmids)} papers")


full-text check: 1837 retained, 54 excluded
reason
no full text in PMC XML    54

retained papers by section count:
n_sections
1     48
2     44
3     76
4     96
5    772
6    801

searchable (>=1 of Results/Discussion/Methods/Conclusion): 1765
abstract-only, retained but never searched: 72 (set REQUIRE_RETRIEVAL_SECTION=True to drop them)

FINAL corpus after all criteria: 1837 papers


## Step 4. Archive the raw XML and the PDF

The JSON holds parsed sections; this step keeps the primary documents alongside it.

XML: the same PMC OAI JATS record the section parser consumes, saved verbatim to `xml/`.
Having it on disk means the corpus can be re-parsed later (a parser fix, a new section
type) without refetching anything, and it is the artifact to cite when documenting
extraction.

PDF: fetched from the Europe PMC render endpoint
(`europepmc.org/articles/PMCxxx?pdf=render`). Availability is pre-flighted through the
Europe PMC search API in batches rather than blind-fetched: papers with
`isOpenAccess=Y` and `hasPDF=Y` download; `hasPDF=N` means no PDF exists; `isOpenAccess=N`
means Europe PMC holds a PDF but will not serve it. Outcomes are recorded per paper, and
refusals are carried forward so reruns do not re-attempt them.

The XML is the authoritative text source for the RAG pipeline; the PDF is an archival
copy for human reading, figures and page-level citation, and nothing downstream depends
on it. Set `FETCH_PDF = False` to skip PDFs entirely.


In [10]:
FETCH_PDF = True
EPMC_REST = "https://www.ebi.ac.uk/europepmc/webservices/rest"
PDF_UA = {"User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/120 Safari/537.36")}

# ---- Step 4a: archive raw PMC OAI XML (verbatim, resume-safe) ----
xml_manifest = load_ckpt("step4a_xml.json", {})

# Reuse pass: an XML already on disk (this run or any earlier run directory) is
# linked and recorded instead of refetched.
_reused = 0
for _c in eligible["pmcid"]:
    if not _c or _c in xml_manifest:
        continue
    _dest = XML_DIR / f"{_c}.xml"
    if not _dest.exists():
        for _rd in REUSE_DOC_DIRS:
            _srcf = _rd / "xml" / f"{_c}.xml"
            if _srcf.exists() and _srcf.stat().st_size > 0:
                try:
                    os.link(_srcf, _dest)
                except OSError:
                    _dest.write_bytes(_srcf.read_bytes())
                break
    if _dest.exists():
        _body = _dest.read_bytes()
        xml_manifest[_c] = {"bytes": len(_body), "has_body": b"<body" in _body}
        _reused += 1
if _reused:
    save_ckpt("step4a_xml.json", xml_manifest)
    print(f"XML reused from disk (no download): {_reused}")

todo = [c for c in eligible["pmcid"] if c and c not in xml_manifest]
for n, pmcid in enumerate(todo, 1):
    dest = XML_DIR / f"{pmcid}.xml"
    try:
        num = re.search(r"(\d+)$", pmcid).group(1)
        r = _request("GET", "https://pmc.ncbi.nlm.nih.gov/api/oai/v1/mh/",
                     params={"verb": "GetRecord", "metadataPrefix": "pmc",
                             "identifier": f"oai:pubmedcentral.nih.gov:{num}"})
        body = r.content
        dest.write_bytes(body)
        has_body = b"<body" in body
        xml_manifest[pmcid] = {"bytes": len(body), "has_body": has_body}
    except Exception as exc:
        xml_manifest[pmcid] = {"bytes": 0, "has_body": False, "error": str(exc)[:120]}
    if n % 25 == 0:
        save_ckpt("step4a_xml.json", xml_manifest)
        print(f"  XML: {n}/{len(todo)}", end="\r")
    time.sleep(0.4)
save_ckpt("step4a_xml.json", xml_manifest)
ok_xml = sum(1 for v in xml_manifest.values() if v.get("has_body"))
print(f"\nXML archived: {len(xml_manifest)} files, {ok_xml} contain a <body> "
      f"(no body means embargoed or metadata-only, and no sections downstream)")


XML reused from disk (no download): 1891

XML archived: 1891 files, 1881 contain a <body> (no body means embargoed or metadata-only, and no sections downstream)


In [11]:
# ---- Step 4b: pre-flight PDF availability via Europe PMC (batched) ----
avail = load_ckpt("step4b_pdf_avail.json", {})
todo = [c for c in eligible["pmcid"] if c and c not in avail]
for i in range(0, len(todo), 50):
    batch = todo[i:i+50]
    q = " OR ".join(f"PMCID:{c}" for c in batch)
    try:
        res = get_json(f"{EPMC_REST}/search",
                       {"query": q, "resultType": "core", "format": "json",
                        "pageSize": 100})["resultList"]["result"]
    except Exception as exc:
        print(f"  preflight batch {i} failed: {str(exc)[:80]}"); res = []
    seen = set()
    for rec in res:
        c = rec.get("pmcid")
        if not c: continue
        seen.add(c)
        avail[c] = {"is_oa": rec.get("isOpenAccess") == "Y",
                    "has_pdf": rec.get("hasPDF") == "Y",
                    "license": rec.get("license") or ""}
    for c in batch:
        avail.setdefault(c, {"is_oa": False, "has_pdf": False, "license": "",
                             "note": "not in Europe PMC"})
    save_ckpt("step4b_pdf_avail.json", avail)
    print(f"  preflight: {min(i+50, len(todo))}/{len(todo)}", end="\r")
    time.sleep(0.5)
print()
fetchable = [c for c, v in avail.items() if v.get("is_oa") and v.get("has_pdf")]
print(f"PDF pre-flight: {len(fetchable)} fetchable of {len(avail)} "
      f"({100*len(fetchable)/max(len(avail),1):.0f} percent)")
print(f"  no PDF in Europe PMC (hasPDF=N)      : {sum(1 for v in avail.values() if not v.get('has_pdf'))}")
print(f"  present but not open access (500s)   : {sum(1 for v in avail.values() if v.get('has_pdf') and not v.get('is_oa'))}")


  preflight: 1891/1891
PDF pre-flight: 1825 fetchable of 1891 (97 percent)
  no PDF in Europe PMC (hasPDF=N)      : 59
  present but not open access (500s)   : 7


In [12]:
# ---- Step 4c: download the PDFs (only the pre-flighted ones) ----
pdf_manifest = load_ckpt("step4c_pdf.json", {})

# Reuse pass: a PDF already on disk (this run or any earlier run directory) is
# linked and recorded instead of re-downloaded. Iterates over ALL eligible pmcids,
# not just currently-fetchable ones: a paper can drop out of the Europe PMC search
# (e.g. PMC7951944 on 2026-09-05) while its archived PDF is still perfectly valid.
_reused = 0
for _c in ([c for c in eligible["pmcid"] if c] if FETCH_PDF else []):
    if pdf_manifest.get(_c, {}).get("ok"):
        continue
    _dest = PDF_DIR / f"{_c}.pdf"
    if not _dest.exists():
        for _rd in REUSE_DOC_DIRS:
            _srcf = _rd / "pdf" / f"{_c}.pdf"
            if _srcf.exists() and _srcf.stat().st_size > 0:
                try:
                    os.link(_srcf, _dest)
                except OSError:
                    _dest.write_bytes(_srcf.read_bytes())
                break
    if _dest.exists():
        pdf_manifest[_c] = {"bytes": _dest.stat().st_size, "ok": True, "reused": True}
        _reused += 1
if _reused:
    save_ckpt("step4c_pdf.json", pdf_manifest)
    print(f"PDF reused from disk (no download): {_reused}")

# Also carry forward prior FAILED outcomes (Europe PMC persistently refuses these),
# so a rerun does not re-attempt known-refused PDFs. Delete those entries from
# step4c_pdf.json in this run's directory to force a retry.
if REUSE_PRIOR_RESULTS and REUSE_DOC_DIRS:
    _prior = REUSE_DOC_DIRS[0] / "step4c_pdf.json"
    if _prior.exists():
        _prev_fail = {c: v for c, v in json.loads(_prior.read_text()).items()
                      if c not in pdf_manifest and not v.get("ok")}
        if _prev_fail:
            pdf_manifest.update(_prev_fail)
            save_ckpt("step4c_pdf.json", pdf_manifest)
            print(f"prior refusals carried forward (not re-attempted): {len(_prev_fail)}")

if FETCH_PDF:
    todo = [c for c in fetchable if c not in pdf_manifest
            and not (PDF_DIR / f"{c}.pdf").exists()]
    print(f"PDFs to fetch: {len(todo)} (already on disk: {len(fetchable) - len(todo)})")
    for n, pmcid in enumerate(todo, 1):
        url = f"https://europepmc.org/articles/{pmcid}?pdf=render"
        try:
            r = requests.get(url, headers=PDF_UA, timeout=120, allow_redirects=True)
            if r.status_code == 200 and r.content[:4] == b"%PDF":
                (PDF_DIR / f"{pmcid}.pdf").write_bytes(r.content)
                pdf_manifest[pmcid] = {"bytes": len(r.content), "ok": True}
            else:
                # 404 = no PDF exists; 500 = present but not served (not open access)
                pdf_manifest[pmcid] = {"bytes": 0, "ok": False,
                                       "status": r.status_code,
                                       "note": r.text[:80] if len(r.text) < 200 else ""}
        except Exception as exc:
            pdf_manifest[pmcid] = {"bytes": 0, "ok": False, "error": str(exc)[:120]}
        if n % 20 == 0:
            save_ckpt("step4c_pdf.json", pdf_manifest)
            got = sum(1 for v in pdf_manifest.values() if v.get("ok"))
            print(f"  PDF: {n}/{len(todo)} (ok {got})", end="\r")
        time.sleep(1.0)      # Europe PMC courtesy pacing
    save_ckpt("step4c_pdf.json", pdf_manifest)
    got = sum(1 for v in pdf_manifest.values() if v.get("ok"))
    mb = sum(v.get("bytes", 0) for v in pdf_manifest.values()) / 1e6
    print(f"\nPDFs downloaded: {got} ({mb:.0f} MB) -> {PDF_DIR}")
else:
    print("FETCH_PDF is False, skipping PDF download")

pd.DataFrame([{"pmcid": c,
               "xml_bytes": xml_manifest.get(c, {}).get("bytes", 0),
               "xml_has_body": xml_manifest.get(c, {}).get("has_body", False),
               "pdf_expected": bool(avail.get(c, {}).get("is_oa") and avail.get(c, {}).get("has_pdf")),
               "pdf_bytes": pdf_manifest.get(c, {}).get("bytes", 0),
               "license": avail.get(c, {}).get("license", "")}
              for c in eligible["pmcid"] if c]
             ).to_csv(OUT_DIR / "step4_document_manifest.csv", index=False)
print("manifest -> step4_document_manifest.csv")


PDF reused from disk (no download): 1616
prior refusals carried forward (not re-attempted): 208
PDFs to fetch: 1 (already on disk: 1824)



PDFs downloaded: 1617 (3125 MB) -> /path/to/project/T21_research_assistant_v7/srv/output/ds_include_pipeline_2026-09-05/pdf
manifest -> step4_document_manifest.csv


---
# Part B. INCLUDE funding tag

The tag is binary and comes from one authoritative source: the curated program
spreadsheet `srv/input/INCLUDE_Publications_by_2025-11-14.xlsx` (428 PMIDs). A corpus
paper whose PMID is in that list is INCLUDE-funded; every other paper is not.

`fpp.load_include_lookup()` is the same loader production uses, so the four provenance
fields travel with the tag and match what the ingestion scripts write: Condition, Grant
Number, RFA/PA Number, Fiscal Year.

No other file is a valid source for this tag; in particular, incremental ingestion lists
are cache diffs, not funding registries.


In [13]:
# ---- B1: load the authoritative INCLUDE list ----
INCLUDE_XLSX = SRV / "input" / "INCLUDE_Publications_by_2025-11-14.xlsx"
include_lookup = fpp.load_include_lookup(INCLUDE_XLSX)   # pmid -> condition/grant/rfa/fiscal_year
include_pmids  = set(include_lookup)
print(f"INCLUDE registry: {len(include_pmids)} PMIDs from {INCLUDE_XLSX.name}")

in_corpus = include_pmids & set(retained_pmids)
print(f"  present in this corpus      : {len(in_corpus)}")
print(f"  absent (failed a criterion, or never returned by the query): "
      f"{len(include_pmids - set(retained_pmids))}")


INCLUDE registry: 428 PMIDs from INCLUDE_Publications_by_2025-11-14.xlsx
  present in this corpus      : 175
  absent (failed a criterion, or never returned by the query): 253


---
# Part C. Apply the tag and export

`is_include` is written into every corpus JSON, True or False, with the four provenance fields
copied from the spreadsheet for tagged papers. This is the same shape `embed_and_insert.py`
reads into the Milvus `is_include` / `condition` / `grant_number` / `rfa_pa_number` /
`fiscal_year` columns, so the UI badge and the citation tag work unchanged.


In [14]:
# ---- C1: write the binary INCLUDE tag into the corpus JSONs ----
FIELDS = ("condition", "grant_number", "rfa_pa_number", "fiscal_year")
n_tagged = n_untagged = n_changed = 0
for pmid in retained_pmids:            # survivors of every criterion, nothing else
    f = JSON_DIR / f"publication_metadata_PMID_{pmid}.json"
    d = json.loads(f.read_text())
    rec = include_lookup.get(pmid)
    vals = {"is_include": rec is not None}
    for k in FIELDS:
        vals[k] = (rec or {}).get(k, "") or ""
    changed = any(d.get(k) != v for k, v in vals.items())
    if changed:
        d.update(vals)
        f.write_text(json.dumps(d, ensure_ascii=False, indent=2))
        n_changed += 1
    n_tagged += bool(rec)
    n_untagged += not bool(rec)
print(f"INCLUDE: {n_tagged} tagged, {n_untagged} not tagged ({n_changed} JSON files updated)")


INCLUDE: 175 tagged, 1662 not tagged (0 JSON files updated)


In [15]:
# ---- C2: master table + summary ----
rows = []
for pmid in retained_pmids:
    d = json.loads((JSON_DIR / f"publication_metadata_PMID_{pmid}.json").read_text())
    secs = d.get("sections") or {}
    pmcid = d.get("pmcid") or ""
    rows.append({
        "pmid": str(d.get("pmid")), "pmcid": pmcid, "title": d.get("title"),
        "year": d.get("year"), "type": d.get("type"),
        "is_include": bool(d.get("is_include")),
        "condition": d.get("condition", ""), "grant_number": d.get("grant_number", ""),
        "n_sections": sum(1 for s in TARGET if present(secs.get(s))),
        "n_retrieval": sum(1 for s in RETRIEVAL_SECTIONS if present(secs.get(s))),
        "has_xml": (XML_DIR / f"{pmcid}.xml").exists() if pmcid else False,
        "has_pdf": (PDF_DIR / f"{pmcid}.pdf").exists() if pmcid else False,
    })
master = pd.DataFrame(rows)
assert master["pmid"].is_unique, "duplicate PMIDs in the master table"
master.to_csv(OUT_DIR / "master_table.csv", index=False)
print("Pipeline complete.")
print(f"  corpus papers      : {len(master)}")
print(f"  INCLUDE-funded     : {int(master['is_include'].sum())}")
print(f"  not INCLUDE        : {int((~master['is_include']).sum())}")
print(f"  searchable papers  : {int((master['n_retrieval'] > 0).sum())} "
      f"(have >=1 of Results/Discussion/Methods/Conclusion)")
print(f"  5+ section papers  : {int((master['n_sections'] >= 5).sum())}")
print(f"  with archived XML  : {int(master['has_xml'].sum())}")
print(f"  with archived PDF  : {int(master['has_pdf'].sum())}")
print(f"  outputs in         : {OUT_DIR}")


Pipeline complete.
  corpus papers      : 1837
  INCLUDE-funded     : 175
  not INCLUDE        : 1662
  searchable papers  : 1765 (have >=1 of Results/Discussion/Methods/Conclusion)
  5+ section papers  : 1573
  with archived XML  : 1836
  with archived PDF  : 1569
  outputs in         : /path/to/project/T21_research_assistant_v7/srv/output/ds_include_pipeline_2026-09-05


---
# Methods provenance

What this notebook does, in the order the manuscript describes it.

Identification. The corpus is the PMC search
`"Down syndrome"[MeSH Terms] AND (open_access[Filter]) AND (published_article[Filter]) AND
(pmc_public[Filter])`, run through `esearch` (db=pmc) rather than the website Download button so
it is reproducible without a manual export. Embargoed articles are excluded by the query, and
again structurally, since an embargoed record serves no OAI body and so yields no sections.

Screening. A paper is retained when it has a PMID, is in PMC, is not retracted, is not an
excluded publication type, and PMC serves usable full text for it. MEDLINE PublicationTypeList
excludes case reports, retracted publications, conference proceedings, comments, editorials,
letters, personal narratives, biographies and introductory articles. The DOI of every remaining
paper is screened for retraction through the CrossRef REST API, which incorporates Retraction
Watch; both retraction signals are used, the MEDLINE type tag and the CrossRef record, because
neither is complete alone. Full-text availability is evaluated after the OAI fetch, since a
PMCID in PubMed does not guarantee the OAI record carries a body. Absent MeSH terms and absent
author keywords never exclude a paper. `step2_screening_audit.csv` records the verdict and
reason for every screened record.

Metadata enrichment, for each retained article:

- OpenAlex API: title, publication year, journal (`source`), language, authors, institutional
  affiliations, funders, and topic classification (`topic`, `subfield`, `field`, `domain`).
- NCBI efetch API: MeSH controlled-vocabulary terms, author keywords, and grant details.
- PMC OAI-PMH interface: structured full-text XML, parsed into the six canonical sections
  (Abstract, Introduction, Methods, Results, Discussion, Conclusion). Nested subsections are
  captured recursively and inherit their parent section.
- CrossRef REST API: retraction screening (above), plus publisher, type and citation count.

Archival. The raw JATS XML is saved for every paper and the publisher PDF where Europe PMC
serves one, so extraction can be repeated later without refetching.

Annotation. Only at the end, papers that satisfied every criterion are matched against
`INCLUDE_Publications_by_2025-11-14.xlsx` and labelled INCLUDE-funded or not, carrying
Condition, Grant Number, RFA/PA Number and Fiscal Year. INCLUDE funding never admits a paper
to the corpus.


---
# Handoff to embedding and Milvus

Outputs in `output/ds_include_pipeline_<RUN_DATE>/`, one directory per run:

- `json_cache/` production-schema JSON per paper (parsed sections plus metadata and INCLUDE tag)
- `xml/` raw PMC JATS XML per paper, for reparsing and provenance
- `pdf/` publisher PDF per paper where Europe PMC serves one
- `step4_document_manifest.csv` what exists for each paper, and why anything is missing
- `master_table.csv` and `step2_screening_audit.csv`, the per-paper selection record

Only `json_cache/` feeds the RAG pipeline. The XML and PDF are archival.

Embedding and indexing are performed by `2_embed_and_index_ds_papers.ipynb`, which reads
`master_table.csv` for the selection and reuses existing embedding pickles wherever the
paper's JSON is unchanged.

Because each run lives in its own dated directory, comparing downloads is a directory
diff: `master_table.csv` and `step2_screening_audit.csv` between two run dates show
exactly which papers appeared, disappeared, or changed verdict. To continue an
interrupted run rather than start a new one, set `RUN_DATE` in the setup cell to that
run's date. After any change to the indexed corpus, restart the API and clear the
response cache, then re-check `/health`.
